# 03 Muon Optimizer 白板编程

**题目:** 手写 `Muon` 优化器 (继承 `torch.optim.Optimizer`)

**Muon 算法 (Momentum + Orthogonalization):**

1. 计算动量: `buf = momentum * buf + grad`
2. 对动量做 Newton-Schulz 正交化 (近似 SVD):
   - 将参数 reshape 成 2D 矩阵
   - 迭代 `ns_steps` 次: `X = a * X + b * X @ X.T @ X + c * X @ X.T @ X @ X.T @ X`
   - 系数: `a=3.4445, b=-4.7750, c=2.0315`
3. 用正交化后的矩阵更新参数: `p = p - lr * X`

**核心思想:** 用正交化替代 Adam 的自适应学习率，在矩阵参数上效果更好

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print("ready")

In [ ]:
# ✏️ 在这里写你的代码

class Muon(torch.optim.Optimizer):
    """
    Muon optimizer: Momentum + Newton-Schulz orthogonalization
    参数:
      - lr: 学习率
      - momentum: 动量系数 (默认 0.95)
      - ns_steps: Newton-Schulz 迭代次数 (默认 5)
    """
    def __init__(self, params, lr=0.02, momentum=0.95, ns_steps=5):
        pass  # 👈 写在这里

    @torch.no_grad()
    def step(self, closure=None):
        pass  # 👈 写在这里

In [ ]:
# ✅ 测试
torch.manual_seed(0)
model = nn.Linear(8, 4, bias=False)
opt = Muon(model.parameters(), lr=0.02, momentum=0.95, ns_steps=5)

x = torch.randn(3, 8)
target = torch.randn(3, 4)

# 训练几步，loss 应该下降
losses = []
for _ in range(20):
    loss = F.mse_loss(model(x), target)
    losses.append(loss.item())
    opt.zero_grad()
    loss.backward()
    opt.step()

assert losses[-1] < losses[0], f"loss 没有下降: {losses[0]:.4f} -> {losses[-1]:.4f}"
print("✅ 通过!")
print(f"   loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"   下降比例: {(1 - losses[-1]/losses[0])*100:.1f}%")